# Streaming Triggers and Output Modes: Append, Complete, and Update

**Module**: 09_streaming  
**Notebook**: 03_triggers_and_output_modes.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Deep dive into Triggers (Default, ProcessingTime, Once, AvailableNow) and Output Modes (Append, Complete, Update).
- **Why does this topic exist?**: Matching the correct output mode to your query type is mandatory; invalid combinations fail at query planning.
- **What problem does it solve?**: Attempting to run Complete mode on a non-aggregated stream or Append mode on global aggregations.
- **Where is it used in real-world Data Engineering?**: Production scheduling of hourly micro-batches using Trigger.AvailableNow.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Essential |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | Structured Streaming basics, DataFrames |
| **Required depth** | Working knowledge: can choose a trigger and output mode for a query |
| **Data Engineer** | Select triggers and output modes that match latency and sink requirements. |
| **Senior Data Engineer** | Diagnose unsupported mode/sink combinations and state growth. |
| **Spark specialist** | Reason about append/update/complete semantics and watermark interaction. |
| **Next topic** | `09_streaming/04_watermarks_and_late_data.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

A trigger controls when a micro-batch runs; an output mode controls which rows are written. The combinations
are constrained: file sinks generally require append mode, complete mode rewrites the entire result and is
unsuitable for unbounded state, and update mode emits changed rows. Choosing the wrong pair leads to unsupported
queries or ever-growing output.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Default trigger** | Run as fast as possible | Next batch starts after the previous finishes |
| **Processing-time trigger** | Fixed interval | Best-effort latency, not exact scheduling |
| **AvailableNow** | Process all available data | Then stop; useful for backfills |
| **Append mode** | Only new rows | Required by most file sinks |
| **Update mode** | Changed rows only | Good for upserts to a mutable sink |
| **Complete mode** | Full result table | Rewrites everything; expensive for large state |

### How it works, step by step

1. A trigger fires based on its schedule or availability.
2. The engine reads new source data.
3. It computes the result according to the query and state.
4. The output mode determines which rows are emitted.
5. The sink writes the rows and the checkpoint commits progress.
6. The next trigger repeats.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Output Mode Matrix
- **Append**: Emits only newly inserted rows. (Required for file sinks; disallowed for global non-watermarked aggregations).
- **Complete**: Re-emits the entire updated result table. (Required for global aggregations; unsupported on file sinks).
- **Update**: Emits only the rows updated since the previous trigger.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Triggers_Modes_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
rate_stream = spark.readStream.format("rate").option("rowsPerSecond", 10).load()
print("Rate stream initialized.")

---

## Experiment: Trigger.AvailableNow Execution

- **1. What are we testing?**: Execute a stream using Trigger.AvailableNow() and observe that it shuts down automatically.
- **2. What do we expect?**: Processes all available rows and terminates cleanly.

In [ ]:
q = (
    rate_stream
    .writeStream
    .format("memory")
    .queryName("avail_now_sink")
    .trigger(availableNow=True)
    .start()
)
q.awaitTermination()
print("Query terminated automatically after Trigger.AvailableNow!")
spark.sql("SELECT count(*) as total_rows FROM avail_now_sink").show()

**3. What happened?**: Stream ran to completion and shut down automatically.  
**4. Why did it happen?**: Trigger.AvailableNow processes pending micro-batches and terminates.  
**5. Practical Engineering Lesson**: Use Trigger.AvailableNow for cost-effective incremental batch ETL scheduled via Airflow.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Dashboard with update mode

- **Industry**: Business intelligence
- **Scenario**: A dashboard needs the latest counts per category, not the full history on every refresh.
- **Problem**: Append mode would emit only new rows, while complete mode rewrites the entire result.
- **Why this concept is relevant**: Update mode emits only changed aggregation rows, reducing sink write volume.
- **Implementation idea**: Aggregate by category and write updates to a key-value store that supports upserts.
- **Expected result**: Low-latency dashboard updates with less write amplification.
- **Alternative**: Complete mode to a small result table.
- **Trade-offs**: Complete mode is simple but becomes expensive as the number of groups grows.

### Use Case 2: Backfill with availableNow

- **Industry**: Logistics
- **Scenario**: A team needs to process a backlog of files once and then stop.
- **Problem**: A continuous trigger keeps the cluster running after the backlog is drained.
- **Why this concept is relevant**: `availableNow` processes all currently available data and terminates, which is ideal for scheduled backfills.
- **Implementation idea**: Run the query with `availableNow`, verify the output, and schedule it as a job.
- **Expected result**: A bounded backfill with no idle cluster time.
- **Alternative**: Use a batch job instead.
- **Trade-offs**: A batch job is simpler for bounded data but may not reuse streaming state/checkpoint logic.

---

## Business Value

- Lower sink write volume and cost.
- Predictable backfill behavior.
- Correct mode/sink compatibility.
- Better latency control.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Use append mode for file sinks and immutable outputs.
- Use update mode for mutable sinks that support upserts.
- Avoid complete mode for large, unbounded state.
- Use `availableNow` for scheduled backfills.
- Remember that processing-time triggers are best-effort, not cron.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Compare output modes

**Steps**

1. Run a streaming aggregation with append, update, and complete modes where supported.
2. Observe the rows emitted per batch and the state size.
3. Identify which mode is compatible with a file sink.

**What to observe**: Complete mode rewrites the full result; update mode emits changes; append mode emits new rows only.

**Healthy result**: The chosen mode matches the sink and the state size.

**Problematic result**: The query fails because the mode is unsupported for the sink or aggregation.

**Engineering takeaway**: Output mode is a correctness and cost decision.

---

## Trade-offs and Alternatives

### Advantages

- Flexible latency control
- Multiple output semantics
- Backfill support

### Limitations

- Mode/sink constraints
- Complete mode is expensive
- Triggers are not exact schedules

### When to use it

- Dashboards
- File ingestion
- Backfills

### When not to use it

- Complete mode with unbounded state
- Append mode with non-append sinks
- Assuming exact trigger times

### Alternatives inside Spark

- Triggers
- Output modes
- Checkpoint

### Alternatives outside Spark

- Workflow schedulers
- Streaming databases
- CDC tools

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Fixed Interval Trigger
**Problem Statement**: Configure trigger with processingTime='5 seconds'.  
**Input / Context**: trigger

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Complete Mode Aggregation
**Problem Statement**: Run a streaming group count in Complete mode to memory sink.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Trigger.Once vs AvailableNow
**Problem Statement**: Explain why AvailableNow replaced Once.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### Complete mode state explosion

**Scenario**: A complete-mode aggregation over a high-cardinality key grows until the job fails.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check the number of groups and the state size.
2. Switch to update or append mode with a watermark if the business logic allows.
3. Bound the state by windowing and watermarking.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Fixed Interval Trigger):
```python
print('trigger(processingTime="5 seconds")')
```

#### Solution 2 (Complete Mode Aggregation):
```python
print('rate_stream.groupBy().count().writeStream.outputMode("complete")')
```

#### Solution 3 (Trigger.Once vs AvailableNow):
```python
print('Trigger.Once processed only 1 micro-batch; AvailableNow processes multiple batches until caught up.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Modes**: Append (new rows), Complete (all rows), Update (modified rows).
- **AvailableNow**: Cost-effective serverless batch processing.

### Essential Technical Terms
- `Trigger.AvailableNow`: Process all pending data and stop.
- `OutputMode`: Defines what is emitted to the sink.

### Common Engineering Mistakes
- **Complete mode on File Sink**: File sink does not support rewriting full tables; requires Append mode.

### Interview Questions on This Topic
- **Q: When should you use Trigger.AvailableNow()?**
  - *A*: When you want incremental streaming semantics but run as an hourly or daily scheduled batch to save compute costs.


### More Interview Questions

- **Q: Which output mode do file sinks generally require?**
  - *A*: Append mode, because file sinks write new files and cannot rewrite previous output efficiently.
- **Q: What does `availableNow` do?**
  - *A*: It processes all data currently available and then stops, making it useful for scheduled or backfill runs.

### What to Study Next
Open 04_watermarks_and_late_data.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')